## Objectif du projet

Ce projet s’inscrit dans une démarche de Data Engineering et Data Analysis appliquée au dataset Olist e-commerce.

L’objectif principal est de construire un pipeline ETL complet permettant :
- d’intégrer plusieurs sources de données hétérogènes
- de nettoyer et transformer les données brutes
- de créer un dataset unifié et exploitable
- d’effectuer des analyses décisionnelles basées sur les données

Ce travail permet ensuite d’extraire des indicateurs clés de performance (KPI) liés aux ventes, aux clients et à la logistique.

## Phase d’initialisation

Cette cellule correspond à la phase de préparation de l’environnement de travail.

In [47]:
# ─────────────────────────────────────────────
# CELLULE 1 — Imports
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# Style global
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"]   = "#f8f8f8"
plt.rcParams["font.family"]      = "DejaVu Sans"
sns.set_palette("husl")

print("Imports OK")


Imports OK


## Chargement des données

Cette étape consiste à charger les différents fichiers CSV du dataset Olist dans des DataFrames pandas afin de préparer les données pour l’analyse et le processus ETL.

In [62]:
# ─────────────────────────────────────────────
# CELLULE 2 — Chargement des 9 fichiers CSV
# ─────────────────────────────────────────────

DATA_PATH = "/"

print("Chargement des fichiers...\n")

datasets = {
    "orders":      pd.read_csv(DATA_PATH + "olist_orders_dataset.csv",
                                parse_dates=["order_purchase_timestamp",
                                             "order_approved_at",
                                             "order_delivered_carrier_date",
                                             "order_delivered_customer_date",
                                             "order_estimated_delivery_date"]),

    "items":       pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv",
                                parse_dates=["shipping_limit_date"]),


    "customers":   pd.read_csv(DATA_PATH + "olist_customers_dataset.csv"),


    "products":    pd.read_csv(DATA_PATH + "olist_products_dataset.csv"),



}

for name, df in datasets.items():
    print(f"   {name:<12} → {df.shape[0]:>7,} lignes   {df.shape[1]:>2} colonnes")

print(f"\n Total lignes (tous fichiers combinés) : {sum(df.shape[0] for df in datasets.values()):,}")



Chargement des fichiers...

   orders       →  99,441 lignes    8 colonnes
   items        → 112,650 lignes    7 colonnes
   customers    →  99,441 lignes    5 colonnes
   products     →  32,951 lignes    9 colonnes

 Total lignes (tous fichiers combinés) : 344,483


## Audit des datasets

Cette étape consiste à analyser chaque dataset afin de comprendre sa structure, sa qualité et ses caractéristiques principales (dimensions, doublons, valeurs manquantes et types de données) avant toute opération de transformation.

In [63]:
# ─────────────────────────────────────────────
# CELLULE 4 — Fonction d'audit par dataset
# ─────────────────────────────────────────────
def audit_dataset(df, name):
    """Affiche un résumé complet d'un dataframe."""
    print(f"\n{'='*60}")
    print(f"   DATASET : {name.upper()}")
    print(f"{'='*60}")
    print(f"  Dimensions  : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    print(f"  Doublons    : {df.duplicated().sum():,}")
    print()

    # Tableau colonnes
    summary = pd.DataFrame({
        "Type"      : df.dtypes,
        "Nulls"     : df.isnull().sum(),
        "Nulls %"   : (df.isnull().mean() * 100).round(1),
        "Uniques"   : df.nunique(),
        "Exemple"   : df.iloc[0],
    })
    print(summary.to_string())
    print()

# Lancer l'audit sur les 5 datasets principaux
for name in ["orders", "customers", "items", "products"]:
    audit_dataset(datasets[name], name)




   DATASET : ORDERS
  Dimensions  : 99,441 lignes × 8 colonnes
  Doublons    : 0

                                         Type  Nulls  Nulls %  Uniques                           Exemple
order_id                               object      0      0.0    99441  e481f51cbdc54678b7cc49136f2d6af7
customer_id                            object      0      0.0    99441  9ef432eb6251297304e76186b10a928d
order_status                           object      0      0.0        8                         delivered
order_purchase_timestamp       datetime64[ns]      0      0.0    98875               2017-10-02 10:56:33
order_approved_at              datetime64[ns]    160      0.2    90733               2017-10-02 11:07:15
order_delivered_carrier_date   datetime64[ns]   1783      1.8    81018               2017-10-04 19:55:00
order_delivered_customer_date  datetime64[ns]   2965      3.0    95664               2017-10-10 21:25:13
order_estimated_delivery_date  datetime64[ns]      0      0.0      459       

##  Analyse des valeurs manquantes — ORDERS

Le dataset Orders contient très peu de valeurs manquantes.  
Ces valeurs ne sont pas des erreurs de données mais reflètent le cycle réel de vie d'une commande :

- Une commande peut ne pas être encore approuvée
- Une commande peut ne pas être encore expédiée
- Une commande peut ne pas être encore livrée

Ainsi, les valeurs manquantes sont **liées au statut de la commande** et non à une mauvaise qualité des données.

In [64]:
# Taux de valeurs manquantes dans Orders

orders_missing = datasets["orders"].isnull().sum().to_frame("missing_values")
orders_missing["missing_%"] = (orders_missing["missing_values"] / len(datasets["orders"]) * 100).round(2)

orders_missing

,missing_values,missing_%
order_id,0,0.00
customer_id,0,0.00
order_status,0,0.00
order_purchase_timestamp,0,0.00
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98
order_estimated_delivery_date,0,0.00


##  Analyse des valeurs manquantes — PRODUCTS

Le dataset Products contient environ 1.9% de valeurs manquantes.

Ces valeurs concernent :
- catégorie produit
- caractéristiques descriptives
- dimensions et poids

Ces données sont importantes pour l’analyse produit et doivent être traitées sans perte d’information.

In [65]:
#Code — analyse des NaN
products_missing = datasets["products"].isnull().sum().to_frame("missing_values")
products_missing["missing_%"] = (products_missing["missing_values"] / len(datasets["products"]) * 100).round(2)

products_missing

,missing_values,missing_%
product_id,0,0.00
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


## Traitement des valeurs manquantes

Une stratégie hybride est appliquée :

- Ajout de variables indicatrices (missing flags)
- Imputation des variables catégorielles par "unknown"
- Imputation des variables numériques par la médiane par catégorie

 Dans la phase d’analyse exploratoire et business, les valeurs imputées (notamment "unknown") sont exclues afin d’éviter tout biais et garantir des résultats fiables et représentatifs.

In [66]:
# 1. Missing flags (traçabilité)
datasets["products"]["missing_category"] = datasets["products"]["product_category_name"].isnull().astype(int)
datasets["products"]["missing_weight"] = datasets["products"]["product_weight_g"].isnull().astype(int)

# 2. Catégoriel
datasets["products"]["product_category_name"] = datasets["products"]["product_category_name"].fillna("unknown")

# 3. Colonnes numériques
num_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in num_cols:
    datasets["products"][col] = datasets["products"].groupby("product_category_name")[col].transform(
        lambda x: x.fillna(x.median())
    )

## Analyse des doublons

Cette étape permet de détecter et quantifier les lignes dupliquées dans chaque dataset afin d’évaluer la qualité des données avant le processus de nettoyage et de transformation.

In [67]:
# ─────────────────────────────────────────────
# Analyse des doublons dans chaque dataset
# ─────────────────────────────────────────────

for name, df in datasets.items():

    print("\n" + "="*60)
    print(f"DATASET : {name.upper()}")
    print("="*60)

    # Nombre de doublons
    nb_duplicates = df.duplicated().sum()

    print(f"Nombre total de lignes : {len(df):,}")
    print(f"Nombre de doublons     : {nb_duplicates:,}")




DATASET : ORDERS
Nombre total de lignes : 99,441
Nombre de doublons     : 0

DATASET : ITEMS
Nombre total de lignes : 112,650
Nombre de doublons     : 0

DATASET : CUSTOMERS
Nombre total de lignes : 99,441
Nombre de doublons     : 0

DATASET : PRODUCTS
Nombre total de lignes : 32,951
Nombre de doublons     : 0


## Jointure des données — Orders & Customers

Dans cette étape, nous commençons le processus de jointure des différents datasets afin de construire une vue unifiée des données.

La première jointure consiste à combiner les informations des commandes avec celles des clients en utilisant la clé `customer_id`, afin d’enrichir les données des commandes avec les caractéristiques clients.

In [55]:
orders_customers = pd.merge(
    datasets["orders"],
    datasets["customers"],
    on="customer_id",
    how="left"
)

print(orders_customers.shape)

(99441, 12)


## Jointure des données — Orders & Items


La deuxième jointure consiste à relier les informations des commandes avec les articles commandés en utilisant la clé `order_id`, afin d’enrichir les données avec les détails des produits achetés et les informations liées à chaque item.

Nous avons choisi une jointure interne (INNER JOIN) afin de ne conserver que les commandes ayant au moins un item associé, garantissant ainsi la cohérence et la fiabilité du dataset final.

In [56]:
orders_items = pd.merge(
    orders_customers,
    datasets["items"],
    on="order_id",
    how="inner"
)

print(orders_items.shape)

(112650, 18)


## Jointure des données — Orders & Products

La jointure est réalisée entre les commandes contenant des items et le dataset des produits en utilisant la clé `product_id`, afin d’ajouter les caractéristiques des produits (catégorie, dimensions, poids, etc.) à chaque ligne de commande.

Nous avons utilisé une jointure gauche (LEFT JOIN) afin de conserver toutes les commandes avec items, même si certaines informations produit sont manquantes.

In [31]:
orders_products = pd.merge(
    orders_items,
    datasets["products"],
    on="product_id",
    how="left"
)

print(orders_products.shape)

(112650, 26)


## Dataset final — orders_products

Après l’ensemble des étapes de jointure et de transformation, nous obtenons le dataset final .

Ce dataset regroupe l’ensemble des informations relatives aux commandes, aux clients, aux articles et aux produits dans une seule table unifiée, prête pour l’analyse et l’extraction des indicateurs métiers.

In [70]:
final_df=orders_products

##  Création du dictionnaire de données

Afin de mieux comprendre la structure du dataset final, un dictionnaire de données a été construit.

Il permet de fournir une description détaillée de chaque colonne, ainsi que des informations sur le type de données et la qualité des données (valeurs manquantes).

In [75]:
import pandas as pd

data_dictionary = pd.DataFrame({
    "column_name": final_df.columns,
    "data_type": final_df.dtypes.astype(str),

})

descriptions = {
    "order_id": "Identifiant unique de la commande",
    "customer_id": "Identifiant du client pour la commande",
    "order_status": "Statut de la commande (delivered, shipped, etc.)",
    "order_purchase_timestamp": "Date et heure d'achat",
    "order_approved_at": "Date d'approbation de la commande",
    "order_delivered_carrier_date": "Date de remise au transporteur",
    "order_delivered_customer_date": "Date de livraison au client",
    "order_estimated_delivery_date": "Date estimée de livraison",
    "customer_unique_id": "Identifiant unique du client (multi-commandes)",
    "customer_zip_code_prefix": "Code postal du client",
    "customer_city": "Ville du client",
    "customer_state": "État du client",
    "order_item_id": "Numéro de l'article dans la commande",
    "product_id": "Identifiant du produit",
    "seller_id": "Identifiant du vendeur",
    "shipping_limit_date": "Date limite d'expédition",
    "price": "Prix du produit",
    "freight_value": "Coût de livraison",
    "product_category_name": "Catégorie du produit",
    "product_name_lenght": "Longueur du nom du produit",
    "product_description_lenght": "Longueur de la description",
    "product_photos_qty": "Nombre de photos du produit",
    "product_weight_g": "Poids du produit en grammes",
    "product_length_cm": "Longueur du produit",
    "product_height_cm": "Hauteur du produit",
    "product_width_cm": "Largeur du produit",

}

data_dictionary["description"] = data_dictionary["column_name"].map(descriptions)

data_dictionary

,column_name,data_type,description
order_id,order_id,object,Identifiant unique de la commande
customer_id,customer_id,object,Identifiant du client pour la commande
order_status,order_status,object,"Statut de la commande (delivered, shipped, etc.)"
order_purchase_timestamp,order_purchase_timestamp,datetime64[ns],Date et heure d'achat
order_approved_at,order_approved_at,datetime64[ns],Date d'approbation de la commande
order_delivered_carrier_date,order_delivered_carrier_date,datetime64[ns],Date de remise au transporteur
order_delivered_customer_date,order_delivered_customer_date,datetime64[ns],Date de livraison au client
order_estimated_delivery_date,order_estimated_delivery_date,datetime64[ns],Date estimée de livraison
customer_unique_id,customer_unique_id,object,Identifiant unique du client (multi-commandes)
customer_zip_code_prefix,customer_zip_code_prefix,int64,Code postal du client


Sauvegarder le dataset final

In [81]:
final_df.to_csv("final_olist_dataset.csv", index=False)